In [1]:
import kagglehub, shutil, os

src = kagglehub.dataset_download("salader/dogsvscats")
dst = "/content/dogs_vs_cats"

if os.path.exists(dst):
    shutil.rmtree(dst)

shutil.copytree(src, dst)

print("Dataset ready at:", dst)

Using Colab cache for faster access to the 'dogsvscats' dataset.
Dataset ready at: /content/dogs_vs_cats


In [2]:
import tensorflow
from tensorflow import keras
from keras import Sequential ,layers
from keras.layers import Dense, Flatten
from keras.applications.vgg16 import VGG16

In [3]:
conv_base = VGG16(
    weights='imagenet',
    include_top=False,
    input_shape=(150, 150, 3)
)

58889256/58889256 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [4]:
conv_base.summary()

Model: "vgg16"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv1 (Conv2D)           │ (None, 150, 150, 64)   │         1,792 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_conv2 (Conv2D)           │ (None, 150, 150, 64)   │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block1_pool (MaxPooling2D)      │ (None, 75, 75, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv1 (Conv2D)           │ (None, 75, 75, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_conv2 (Conv2D)           │ (None, 75, 75, 128)    │       147,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block2_pool (MaxPooling2D)      │ (None, 37, 37, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv1 (Conv2D)           │ (None, 37, 37, 256)    │       295,168 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv2 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_conv3 (Conv2D)           │ (None, 37, 37, 256)    │       590,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block3_pool (MaxPooling2D)      │ (None, 18, 18, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv1 (Conv2D)           │ (None, 18, 18, 512)    │     1,180,160 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv2 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_conv3 (Conv2D)           │ (None, 18, 18, 512)    │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block4_pool (MaxPooling2D)      │ (None, 9, 9, 512)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv1 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv2 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_conv3 (Conv2D)           │ (None, 9, 9, 512)      │     2,359,808 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ block5_pool (MaxPooling2D)      │ (None, 4, 4, 512)      │             0 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 14,714,688 (56.13 MB)

 Trainable params: 14,714,688 (56.13 MB)

 Non-trainable params: 0 (0.00 B)

In [5]:
model = Sequential()

model.add(conv_base)
model.add(Flatten())
model.add(Dense(256,"relu"))
model.add(Dense(1,"sigmoid"))

In [6]:
conv_base.trainable = False

In [7]:
train_ds = keras.utils.image_dataset_from_directory(
    directory = "/content/dogs_vs_cats/train",
    batch_size = 32,
    subset = "training",
    validation_split = 0.2,
    image_size = (150,150),
    seed = 42
)

val_ds = keras.utils.image_dataset_from_directory(
    directory = "/content/dogs_vs_cats/train",
    batch_size = 32,
    subset = "validation",
    validation_split = 0.2,
    image_size = (150,150),
    seed = 42
)
test_ds = keras.utils.image_dataset_from_directory(
    directory = "/content/dogs_vs_cats/test",
    batch_size = 32,
    image_size = (150,150)
)

Found 20000 files belonging to 2 classes.
Using 16000 files for training.
Found 20000 files belonging to 2 classes.
Using 4000 files for validation.
Found 5000 files belonging to 2 classes.


In [8]:
def process(image, label):
    image = tensorflow.cast(image/255., tensorflow.float32)
    return image, label

train_ds = train_ds.map(process).prefetch(tensorflow.data.AUTOTUNE)
val_ds   = val_ds.map(process).prefetch(tensorflow.data.AUTOTUNE)

In [9]:
model.compile(
    optimizer = "adam",
    loss = "binary_crossentropy",
    metrics = ["accuracy"]
)

In [10]:
model.fit(train_ds, validation_data = val_ds ,epochs=10)

Epoch 1/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 69s 120ms/step - accuracy: 0.8533 - loss: 0.3737 - val_accuracy: 0.9158 - val_loss: 0.2056
Epoch 2/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 61s 122ms/step - accuracy: 0.9203 - loss: 0.1954 - val_accuracy: 0.9007 - val_loss: 0.2330
Epoch 3/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 64s 127ms/step - accuracy: 0.9355 - loss: 0.1657 - val_accuracy: 0.9013 - val_loss: 0.2455
Epoch 4/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 83s 129ms/step - accuracy: 0.9389 - loss: 0.1495 - val_accuracy: 0.9112 - val_loss: 0.2481
Epoch 5/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 64s 127ms/step - accuracy: 0.9526 - loss: 0.1201 - val_accuracy: 0.9082 - val_loss: 0.2509
Epoch 6/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 64s 127ms/step - accuracy: 0.9670 - loss: 0.0875 - val_accuracy: 0.9153 - val_loss: 0.2676
Epoch 7/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 64s 127ms/step - accuracy: 0.9747 - loss: 0.0707 - val_accuracy: 0.9097 - val_loss: 0.2917
Epoch 8/10
500/500 ━━━━━━━━━━━━━━━━━━━━ 71s 143ms/step - accuracy: 0.9826 - loss: 0

In [11]:
model.evaluate(test_ds)

157/157 ━━━━━━━━━━━━━━━━━━━━ 20s 129ms/step - accuracy: 0.8856 - loss: 34.1976


[36.33498001098633, 0.88919997215271]